# 유지형 되뇌기(A) — 데이터 준비

- **input** = 원문 문서 전체(문장을 원래 순서대로 이어붙인 것)
- **target** = `extractive` 인덱스로 선택된 문장들을 **원문 등장 순서 그대로**
  이어붙인 텍스트(원본 JSON의 `extractive`는 중요도 순 `[5,4,2]`처럼 정렬 안 돼
  있어서 반드시 오름차순 정렬 후 이어붙여야 함 — 실제 데이터로 확인함)

`data/raw/aihub_summary/`에 AIHub 데이터가 이미 도착해 있어(법률문서/사설·잡지/
문서요약 3개 프로젝트, 각각 86MB~1.16GB) 더미 pseudo-label 경로 없이 바로 실 데이터로 진행한다. 이번 노트북은 그중 가장 작은 **법률문서 프로젝트**(86MB, 24,329문서)로 파일럿을 돌린다 — 코드는 다른 두 파일에도 그대로 재사용 가능하게 파일 경로만 바꾸면 되도록 짰다.

In [1]:
import json
import random
from pathlib import Path

import sys

root = Path.cwd()
while not (root / "src").exists() and root != root.parent:
    root = root.parent
sys.path.insert(0, str(root))

from src.notebook_setup import setup_project

setup_project()

import datasets
import pandas as pd
from transformers import AutoTokenizer


project root: /Users/lucyroh/Desktop/STUDY/Data Projects/rehearse-then-recall
.env 로드: 성공 ✅
NVIDIA_NIM_API_KEY: 설정됨 ✅


/Users/lucyroh/Desktop/STUDY/Data Projects/rehearse-then-recall/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. AIHub JSON 로드 + 스키마 확인

스키마(실제 파일 확인함): `{"name", "delivery_date", "documents": [{"id",
"category", "size", "char_count", "text": [[{"index","sentence","highlight_indices"},
...], ...], "extractive": [idx, ...], "abstractive": [...]}]}`.

`text`는 문단 리스트(각 문단은 문장 dict 리스트)인데, 문장의 `index`는 문서
**전체에 걸친 전역 인덱스**다(문단이 바뀌어도 리셋 안 됨 — 실제 데이터로
확인함). 그래서 문단 구분 없이 `index` 순서로 펼쳐서 이어붙이면 원문 전체가
된다.

In [2]:
AIHUB_FILE = "data/raw/aihub_summary/Training/train_original 2.json"  # 법률문서 프로젝트(가장 작음, 86MB)

with open(AIHUB_FILE, encoding="utf-8") as f:
    aihub_data = json.load(f)

documents = aihub_data["documents"]
print(f"데이터셋: {aihub_data['name']}, 문서 수: {len(documents)}")

sample_doc = documents[0]
print("\n샘플 문서 키:", list(sample_doc.keys()))
print("문단 수:", len(sample_doc["text"]))
print("extractive(정렬 전):", sample_doc["extractive"])


데이터셋: 법률문서 프로젝트, 문서 수: 24329

샘플 문서 키: ['id', 'category', 'size', 'char_count', 'publish_date', 'title', 'text', 'annotator_id', 'document_quality_scores', 'extractive', 'abstractive']
문단 수: 1
extractive(정렬 전): [5, 4, 2]


## 2. (input, target) 쌍 생성

`extractive`를 오름차순 정렬 후 해당 문장들을 이어붙여 `target`을 만든다.
`extractive`가 비어 있는 문서(추출요약 라벨이 없는 경우)는 학습 신호가 없으므로
제외한다.

In [3]:
def build_pair(doc: dict) -> dict | None:
    sentence_by_index = {s["index"]: s["sentence"] for para in doc["text"] for s in para}
    if not doc["extractive"] or not sentence_by_index:
        return None

    input_text = " ".join(sentence_by_index[i] for i in sorted(sentence_by_index))
    target_text = " ".join(sentence_by_index[i] for i in sorted(doc["extractive"]) if i in sentence_by_index)
    if not target_text:
        return None

    return {
        "id": doc["id"],
        "size": doc.get("size"),
        "input_text": input_text,
        "target_text": target_text,
    }


pairs = [p for p in (build_pair(doc) for doc in documents) if p is not None]
print(f"유효한 (input, target) 쌍: {len(pairs)} / {len(documents)}")

pairs_df = pd.DataFrame(pairs)
pairs_df[["input_text", "target_text"]].head(3)


유효한 (input, target) 쌍: 24329 / 24329


,input_text,target_text
0,원고가 소속회사의 노동조합에서 분규가 발생하자 노조활동을 구실로 정상적인 근무를 해...,노동조합규약에 동 조합장의 직무를 대행할 자를 규정해 두고 있음에도 원고 자신이 주...
1,수출입업체인 원고가 의류제품을 제조ㆍ수출함에 있어 같은 그룹내 종합무역상사인 소외 ...,수출입업체인 원고가 의류제품을 제조ㆍ수출함에 있어 같은 그룹내 종합무역상사인 소외 ...
2,가등기담보권자가 제소전 화해조항에 따라 자기 명의로 소유권이전의 본등기를 경료한 후...,가등기담보권자가 제소전 화해조항에 따라 자기 명의로 소유권이전의 본등기를 경료한 후...


## 3. 서브샘플 + 토큰화

전체 24k문서를 다 쓰지 않고 파일럿 규모(`MAX_EXAMPLES`)로 무작위 서브샘플링

`max_length`는 09_청킹 구현 가이드 §5에서 추정치였던 어절→토큰 비율을
`paust/pko-t5-small` 토크나이저로 실측한 결과(어절당 약 2.79토큰, 추정치
1.5~2보다 높음)를 반영해 정했다 — input은 pko-t5 공식 학습 설정(1300)을
그대로 쓰고, target(추출요약이라 input보다 항상 짧음)은 512로 넉넉히 잡는다.

In [4]:
MAX_EXAMPLES = 3000
VAL_RATIO = 0.1
INPUT_MAX_LENGTH = 1300
TARGET_MAX_LENGTH = 512
SEED = 42

random.seed(SEED)
sampled = pairs if len(pairs) <= MAX_EXAMPLES else random.sample(pairs, MAX_EXAMPLES)
random.shuffle(sampled)

n_val = max(1, int(len(sampled) * VAL_RATIO))
val_pairs = sampled[:n_val]
train_pairs = sampled[n_val:]
print(f"train: {len(train_pairs)}개, val: {len(val_pairs)}개")

tokenizer = AutoTokenizer.from_pretrained("paust/pko-t5-small")


def tokenize_pairs(pairs: list[dict]) -> datasets.Dataset:
    inputs = tokenizer(
        [p["input_text"] for p in pairs],
        max_length=INPUT_MAX_LENGTH,
        truncation=True,
    )
    targets = tokenizer(
        [p["target_text"] for p in pairs],
        max_length=TARGET_MAX_LENGTH,
        truncation=True,
    )
    return datasets.Dataset.from_dict(
        {
            "input_ids": inputs["input_ids"],
            "attention_mask": inputs["attention_mask"],
            "labels": targets["input_ids"],
        }
    )


train_dataset = tokenize_pairs(train_pairs)
val_dataset = tokenize_pairs(val_pairs)
print(train_dataset)


train: 2700개, val: 300개


Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 2700
})


## 4. 저장

토큰화된 데이터셋과(재현/디버깅용으로) 원본 텍스트 쌍을 함께 저장한다.
`experiments/rehearsal_maintenance_small/`(학습 노트북)에서 그대로 불러 쓴다.

In [5]:
OUT_DIR = Path("data/processed/rehearsal_maintenance")
OUT_DIR.mkdir(parents=True, exist_ok=True)

train_dataset.save_to_disk(str(OUT_DIR / "train"))
val_dataset.save_to_disk(str(OUT_DIR / "val"))

pd.DataFrame(train_pairs).to_csv(OUT_DIR / "train_pairs_raw.csv", index=False, encoding="utf-8-sig")
pd.DataFrame(val_pairs).to_csv(OUT_DIR / "val_pairs_raw.csv", index=False, encoding="utf-8-sig")

print(f"저장 완료: {OUT_DIR}")
print(f"  train_dataset: {len(train_dataset)}행")
print(f"  val_dataset: {len(val_dataset)}행")



Saving the dataset (0/1 shards):   0%|          | 0/2700 [00:00<?, ? examples/s]

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)

Saving the dataset (1/1 shards): 100%|██████████| 2700/2700 [00:00<00:00, 161591.00 examples/s]


Saving the dataset (1/1 shards): 100%|██████████| 2700/2700 [00:00<00:00, 157223.07 examples/s]


Saving the dataset (0/1 shards):   0%|          | 0/300 [00:00<?, ? examples/s]


Saving the dataset (1/1 shards): 100%|██████████| 300/300 [00:00<00:00, 102358.35 examples/s]


Saving the dataset (1/1 shards): 100%|██████████| 300/300 [00:00<00:00, 92283.92 examples/s] 

저장 완료: data/processed/rehearsal_maintenance
  train_dataset: 2700행
  val_dataset: 300행


## 정리

- 사용한 원본: `법률문서 프로젝트`(86MB, 24,329문서 중 유효 쌍만 필터링 후
  `MAX_EXAMPLES`(기본 3,000)개 서브샘플)
- 나머지 두 AIHub 파일(`사설/잡지 문서 프로젝트` 313MB, `문서요약 프로젝트`
  1.16GB)은 이번 파일럿에서는 안 썼다 — `AIHUB_FILE` 경로만 바꾸면 이 노트북을
  그대로 재사용해 스케일업할 수 있다. 단, 1.16GB 파일은 `json.load`로 통째로
  읽으면 메모리를 상당히 쓰니(로컬 16GB RAM 기준) 스트리밍 파서(`ijson` 등)로
  바꾸는 걸 고려할 것.
- `max_length` 추정치(09가이드 §5)를 실측치(어절당 2.79토큰)로 갱신 — 원래
  가정(1.5~2토큰/어절)보다 상당히 높았다. `configs/chunking.yaml`의
  `max_words=350` 자체를 재조정할지는 이 노트북 범위 밖(청킹은 "완성" 상태)이라
  건드리지 않았지만, 다음에 청킹 설정을 다시 볼 때 참고할 것.